# **Generate video analysis after network training, evaluation, and refinement on the GUI**

In [1]:
import deeplabcut
from pathlib import Path
import time
import re
import os

Loading DLC 3.0.0rc13...


d:\Anaconda3\envs\DEEPLABCUT\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Final model

path_config_file = r"C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\trainingSessions\trainingSessions_topView_DLC\trainingAggression_trainingSessions_topView-MiguelRocha-2026-02-03\config.yaml"
shuffle = 3
iteration = 3

'''
deeplabcut.export_model(path_config_file,
                        shuffle = shuffle,
                        iteration = iteration,
                        overwrite = False)
'''

'\ndeeplabcut.export_model(path_config_file,\n                        shuffle = shuffle,\n                        iteration = iteration,\n                        overwrite = False)\n'

In [3]:
videos_path = Path(r"C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\trainingSessions\trainingSessions")
avis = [f for f in videos_path.rglob("*topView_comp.avi")]
filtered_avis = [f for f in avis if "preTrain" not in str(f) and not any(f.parent.rglob("*topView_DLCtracking_pcutoff*"))] # Filters through the preTrain videos and already processed videos

print(f"Number of videos to analyze: {len(filtered_avis)}")

Number of videos to analyze: 251


In [4]:
# Filter the last day videos for every adaptive mice. The Maladaptive mice are filtered for the last day of the first week

additionalFilter = True

if (additionalFilter):
    best = {}  # mouse_id -> (day_int, path)

    for p in avis:
        s = str(p)

        m_mouse = re.search(r"mouse(\d+)", s)
        m_day   = re.search(r"[\\/](Day)(\d+)", s)  # ...\Day02\...

        if not (m_mouse and m_day):
            continue

        mouse_id = m_mouse.group(1)
        day = int(m_day.group(2))

        # If path contains "Maladaptive", only keep single-digit Days
        if "Maladaptive" in s and day >= 10:
            continue

        if (mouse_id not in best) or (day > best[mouse_id][0]):
            best[mouse_id] = (day, p)

    last_day_paths = [t[1] for t in best.values()]
    last_day_by_mouse = {mid: t[1] for mid, t in best.items()}


In [5]:
videos = last_day_paths

pcutoff = 0.70
tracking_method = "transformer"

for v in videos:

    start_T = time.time()

    print("\n")
    print(f"============================= {v.name} =============================")

    destFolder = Path(f"{v.parent}/topView_DLCtracking_pcutoff_{pcutoff}_{tracking_method}")

    try:
        os.mkdir(destFolder)
    except FileExistsError:
        pass
    except OSError as e:
        print(f"Could not create folder {destFolder}: {e}")
        raise

    deeplabcut.analyze_videos(path_config_file,
                              v,
                              videotype=".avi",
                              shuffle=shuffle,
                              auto_track = True,
                              identity_only = True, # Implant keypoints are enough to distinguish mice. Bypasses identity estimation. This was possible by specifying keypoints only present in one mouse (implant -> Resident) during annotations
                              dynamic = (False, .5,10),
                              destfolder=destFolder)

    
    deeplabcut.transformer_reID(path_config_file,
                                str(v),
                                shuffle=shuffle,
                                videotype="avi",
                                track_method="skeleton",
                                n_triplets=6000,
                                train_epochs=200,
                                n_tracks = 2,
                                destfolder=destFolder)
    
                                
    deeplabcut.filterpredictions(path_config_file, 
                                 str(v), 
                                 videotype=".avi", 
                                 shuffle=shuffle,
                                 track_method=tracking_method,
                                 destfolder=destFolder)

    deeplabcut.plot_trajectories(path_config_file, 
                                 str(v), 
                                 shuffle=shuffle, 
                                 filtered = True, 
                                 track_method=tracking_method,
                                 destfolder=destFolder)

    deeplabcut.create_labeled_video(path_config_file, 
                                    str(v), 
                                    videotype='.avi', 
                                    shuffle=shuffle, 
                                    filtered=True, 
                                    fastmode=True, 
                                    save_frames=False, 
                                    keypoints_only=False, 
                                    Frames2plot=None, 
                                    displayedbodyparts='all', 
                                    displayedindividuals='all',
                                    outputframerate=None, 
                                    draw_skeleton=True, 
                                    trailpoints=0, 
                                    displaycropped=False, 
                                    track_method=tracking_method,
                                    dotsize = 3,
                                    destfolder=destFolder,
                                    skeleton_color="red",
                                    confidence_to_alpha = True,
                                    plot_bboxes = False, 
                                    pcutoff = pcutoff,
                                    color_by = "individual")
    
    stop_T = time.time()

    print(f"Processing time: {((stop_T - start_T)/60):.2f} min\n")



============================= mouse975826_Day19_topView_comp.avi =============================
Analyzing videos with C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\trainingSessions\trainingSessions_topView_DLC\trainingAggression_trainingSessions_topView-MiguelRocha-2026-02-03\dlc-models-pytorch\iteration-3\trainingAggression_trainingSessions_topViewFeb3-trainset99shuffle3\train\snapshot-best-070.pt
Using scorer: DLC_Resnet101_trainingAggression_trainingSessions_topViewFeb3shuffle3_snapshot_best-70
Starting to analyze C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\trainingSessions\trainingSessions\20250825_mouse975826_trainingSessions\Day19\mouse975826_Day19_topView_comp.avi
Video metadata: 
  Overall # of frames:    64586
  Duration of video [s]:  1291.72
  fps:                    50.0
  resolution:             w=552, h=292

Running pose prediction with batch size 8


  0%|          | 31/64586 [00:00<06:44, 159.74it/s]d:\Anaconda3\envs\DEEPLABCUT\Lib\site-packages\deeplabcut\pose_estimation_pytorch\data\postprocessor.py:514: RuntimeWarning: invalid value encountered in cast
  heatmap_indices = np.rint(individual_keypoints).astype(int)
100%|██████████| 64586/64586 [23:44<00:00, 45.34it/s]


Processing...  C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\trainingSessions\trainingSessions\20250825_mouse975826_trainingSessions\Day19\mouse975826_Day19_topView_comp.avi
Loading From C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\trainingSessions\trainingSessions\20250825_mouse975826_trainingSessions\Day19\topView_DLCtracking_pcutoff_0.7_transformer\mouse975826_Day19_topView_compDLC_Resnet101_trainingAggression_trainingSessions_topViewFeb3shuffle3_snapshot_best-70.h5


100%|██████████| 64586/64586 [00:12<00:00, 5189.93it/s]


The tracklets were created (i.e., under the hood deeplabcut.convert_detections2tracklets was run). Now you can 'refine_tracklets' in the GUI, or run 'deeplabcut.stitch_tracklets'.
Processing...  C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\trainingSessions\trainingSessions\20250825_mouse975826_trainingSessions\Day19\mouse975826_Day19_topView_comp.avi


...

The tracklets were created (i.e., under the hood deeplabcut.convert_detections2tracklets was run). Now you can 'refine_tracklets' in the GUI, or run 'deeplabcut.stitch_tracklets'.
Processing...  C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\trainingSessions\trainingSessions\20251006_mouse1010820_trainingSessions\Day12\mouse1010820_Day12_topView_comp.avi


100%|██████████| 50/50 [00:00<00:00, 159.95it/s]
d:\Anaconda3\envs\DEEPLABCUT\Lib\site-packages\deeplabcut\refine_training_dataset\stitch.py:941: FutureWarning: Starting with pandas version 3.0 all arguments of to_hdf except for the argument 'path_or_buf' will be keyword-only.
  df.to_hdf(output_name, "tracks", format="table", mode="w")


The videos are analyzed. Now your research can truly start!
You can create labeled videos with 'create_labeled_video'.
If the tracking is not satisfactory for some videos, consider expanding the training set. You can use the function 'extract_outlier_frames' to extract a few representative outlier frames.

Loading C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\trainingSessions\trainingSessions\20251006_mouse1010820_trainingSessions\Day12\mouse1010820_Day12_topView_comp.avi
Duration of video [s]: 1343.90, recorded with 50.00 fps!
Overall # of frames: 67195 found with (before cropping)
Frame dimensions: 552 x 292


 13%|█▎        | 8720/67195 [03:28<23:15, 41.91it/s]


KeyboardInterrupt: 

In [ ]:
'''
import shutil

videos_path = Path(r"C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\trainingSessions\trainingSessions")
avis = [f for f in videos_path.rglob("*topView_DLC*")]

for i in avis:
    shutil.rmtree(i)

'''